# Feature Engineering Notebook


# Setup

In [1]:
#imports
import pandas as pd

In [2]:
df_l = pd.read_csv(".././data/cleaned/cleaned_listings.csv", low_memory=False)
df_s = pd.read_csv(".././data/cleaned/cleaned_sold.csv", low_memory=False)

# Engineering

In [3]:
# fe helper functions
def make_price_ratio(df): 
    df = df.copy()
    df['price_ratio'] = df['ClosePrice'] / df['OriginalListPrice']
    return df

def make_price_per_sqft(df): 
    df = df.copy()
    df['price_per_sqft'] = df['ClosePrice'] / df['LivingArea']
    return df

def make_days_on_market(df):
    df = df.copy()
    df['days_on_market'] = (pd.to_datetime(df['CloseDate']) - pd.to_datetime(df['ListingContractDate'])).dt.days
    return df

def make_yr_month(df):
    df = df.copy()
    df['yr_month'] = pd.to_datetime(df['CloseDate']).dt.to_period('M')
    return df

def make_listing_to_contract_days(df):
    df = df.copy()
    df['listing_to_contract_days'] = (pd.to_datetime(df['PurchaseContractDate']) - pd.to_datetime(df['ListingContractDate'])).dt.days
    return df

def make_contract_to_close_days(df):
    df = df.copy()
    df['contract_to_close_days'] = (pd.to_datetime(df['CloseDate']) - pd.to_datetime(df['PurchaseContractDate'])).dt.days
    return df


In [4]:
# call func for all
def feature_engineering(df):
    # Example feature engineering steps
    df = make_price_ratio(df)
    df = make_price_per_sqft(df)
    df = make_days_on_market(df)
    df = make_yr_month(df)
    df = make_listing_to_contract_days(df)
    df = make_contract_to_close_days(df)

    return df

In [5]:
df_l.head(5)

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,...,BuyerAgencyCompensation,year_month,impossible_measurement_flag,impossible_year_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,null_coords_flag,placeholder_coords_flag,non_cali_coords_flag
0,929000.0,1076194146,NaN,NaN,NaN,NaN,16882 Canyon Lane,Residential,1389.0,929000.0,...,NaN,2024-05,False,False,False,False,False,True,False,False
1,999999.0,1076194026,NaN,NaN,NaN,NaN,8720 S 4th Avenue,Residential,2526.0,999999.0,...,NaN,2024-05,False,False,False,False,False,True,False,False
2,1400000.0,1076193814,NaN,NaN,33.858559,-116.542169,505 E Molino Road,Residential,2256.0,1400000.0,...,NaN,2024-05,False,False,False,False,False,False,False,True
3,4998888.0,1076193812,NaN,NaN,NaN,NaN,3653 Halldale Avenue,ResidentialIncome,NaN,4998888.0,...,NaN,2024-05,False,False,False,False,False,True,False,False
4,549000.0,1076193525,NaN,NaN,NaN,NaN,1736 N Mcdivitt Avenue,Residential,986.0,549000.0,...,NaN,2024-05,False,False,False,False,False,True,False,False


In [6]:
df_le = feature_engineering(df_l)
df_ls = feature_engineering(df_s)

# Pipeline Testing


In [7]:
from idx.components import ingest, preprocessing
from sklearn.pipeline import Pipeline


ingestor = ingest.MLSIngestor(input_path="../data/raw/")
fred_ingestor = ingest.FredMerger()
cleaner = preprocessing.DataCleaner()
flagger = preprocessing.BadDataFlagger()

ingest_test = Pipeline([ 
    ("ingestor", ingestor),
    ("fred_ingestor", fred_ingestor),
    ("cleaner", cleaner),
    ("flagger", flagger)
])


df_sold, df_listings = ingest_test.fit_transform(X=None, y=None)